# FreshPrice AI — Interactive Demo

**QStorePrice AI** is a perishable goods intelligence platform.  
An RL-trained LLM (Qwen-2.5-7B) writes **Operating Briefs** that drive three business engines:

| Engine | What it does | Reward weight |
|--------|-------------|---------------|
| Pricing (r1) | Discounts items before they expire | 40% |
| Farmer (r2) | Accepts/declines supplier offers | 30% |
| Trend (r3)  | Restocks on viral demand signals | 30% |

**WRR (Weekly Waste Recovery Rate)** = revenue from at-risk inventory / cost of at-risk inventory  
Target: ≥ 0.70 across all 5 curriculum scenarios.

---

## Hackathon Theme Coverage
- **Theme #1 (Multi-Agent)**: `MultiStoreFreshPriceEnv` + `ConsumerAgent` — two-sided market with theory-of-mind
- **Theme #2 (Long-Horizon)**: 672-tick / 7-day episodes with sparse WRR signal; long-horizon 28-day mode available
- **Theme #3 (World Modeling)**: External shocks (weather, festivals) + professional store management task
- **Theme #4 (Self-Improvement)**: 5-level adaptive curriculum + self-play negotiation arena

## 1. Install & Import

In [ ]:
# Install dependencies (skip if already installed)
# !pip install -r requirements.txt

import sys
sys.path.insert(0, '.')

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from eval.baselines.random_agent import RandomAgent
from eval.baselines.rule_based_agent import RuleBasedAgent

print('Imports OK')

## 2. Initialize the Environment

In [ ]:
env = FreshPriceEnv(
    scenario=CurriculumScenario.STABLE_WEEK,
    seed=42,
    render_mode='human',
)

obs, info = env.reset()
print(f'Scenario: {info["scenario"]}')
print(f'First engine: {info["engine_type"]}')
print(f'Observation length: {len(obs)} chars')

## 3. View the LLM Observation (What the Agent Sees)

In [ ]:
print('=== OBSERVATION (first 2000 chars) ===')
print(obs[:2000])
print('...' if len(obs) > 2000 else '')

## 4. Sample Operating Brief (What the LLM Writes)

In [ ]:
SAMPLE_BRIEF = '''
## SITUATION
Batch batch_0001 (vegetables): 18.5h to expiry, 22 units, URGENT status.
Batch batch_0003 (dairy): 4.2h to expiry, 8 units, CRITICAL status.
Weather: SUNNY | No active events.

## SIGNAL ANALYSIS
CRITICAL dairy (batch_0003): immediate 50% clearance required — 4h remaining.
URGENT vegetables (batch_0001): 30% discount needed to clear before expiry.
Demand velocity nominal — no trend signals active.

## VIABILITY CHECK
batch_0001 floor=Rs 15.12, 30% discount → Rs 26.25 (above floor). VIABLE.
batch_0003 floor=Rs 26.25, 50% discount → Rs 30.00 (above floor). VIABLE.

## RECOMMENDATION
Apply 30% discount to batch_0001 (vegetables URGENT).
Apply 50% flash sale to batch_0003 (dairy CRITICAL) — prioritise clearance.

## DIRECTIVE
{"engine": "PRICING", "actions": [
  {"batch_id": "batch_0001", "price_multiplier": 0.70, "flash_sale": false},
  {"batch_id": "batch_0003", "price_multiplier": 0.50, "flash_sale": true}
]}

## CONFIDENCE
HIGH
'''.strip()

print(SAMPLE_BRIEF)

## 5. Step Through One Episode with Rule-Based Agent

In [ ]:
agent = RuleBasedAgent()
env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
obs, info = env.reset()

wrr_history = []
reward_history = []
step = 0
done = False

print('Running episode... (84 steps = 7 simulated days)')
while not done:
    action = agent.act(obs, info)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    step += 1
    state = env.state()
    wrr_history.append(state['wrr_so_far'])
    reward_history.append(reward)
    if step % 12 == 0 or done:  # log every ~1 simulated day
        print(f'  Step {step:>3} | Day {step//12+1} | WRR={state["wrr_so_far"]:.3f} | '
              f'Critical={state["critical_batches"]} | Weather={state["weather"]}')

final = info.get('final_reward', {})
print(f'\nEpisode complete!')
print(f'  Final WRR:      {final.get("wrr", 0):.3f}')
print(f'  r1 (pricing):   {final.get("r1_pricing", 0):.3f}')
print(f'  r2 (farmer):    {final.get("r2_farmer", 0):.3f}')
print(f'  r3 (trend):     {final.get("r3_trend", 0):.3f}')
print(f'  CO2 saved:      {final.get("co2_saved_kg", 0):.1f} kg')
print(f'  Food saved:     {final.get("kg_food_saved", 0):.1f} kg')
print(f'  Anti-hack:      {final.get("anti_hack_violations", 0)} violations')

## 6. Plot WRR Progress Over the Episode

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

steps = list(range(1, len(wrr_history) + 1))
days = [s / 12 for s in steps]  # 12 briefs per day

ax1.plot(days, wrr_history, color='#2196F3', linewidth=2)
ax1.axhline(0.70, color='#4CAF50', linestyle='--', linewidth=1.5, label='Target WRR 0.70')
ax1.fill_between(days, wrr_history, alpha=0.15, color='#2196F3')
ax1.set_xlabel('Simulated Day')
ax1.set_ylabel('WRR (Weekly Waste Recovery Rate)')
ax1.set_title('WRR Progress — Rule-Based Agent (STABLE_WEEK)')
ax1.legend()
ax1.set_ylim(0, 1.0)
ax1.grid(alpha=0.3)

ax2.bar(steps, reward_history, color=['#E53935' if r < 0 else '#43A047' for r in reward_history], alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Brief Step')
ax2.set_ylabel('Reward Delta (WRR improvement)')
ax2.set_title('Per-Brief Reward Signal')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eval/plots/demo_episode.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to eval/plots/demo_episode.png')

## 7. Compare Baselines Across Scenarios

In [ ]:
from eval.baselines.run_baselines import run_agent_episodes, aggregate

SCENARIOS = [
    CurriculumScenario.STABLE_WEEK,
    CurriculumScenario.CRISIS_WEEK,
]

agents = [
    ('RandomAgent',    RandomAgent(seed=42)),
    ('RuleBasedAgent', RuleBasedAgent()),
]

print(f"{'Agent':<20} {'Scenario':<16} {'WRR':>7} {'CO2 saved':>12}")
print('-' * 60)
for scenario in SCENARIOS:
    for name, agent in agents:
        results = run_agent_episodes(agent, scenario, n_episodes=2)
        agg = aggregate(results)
        print(f"{name:<20} {scenario.name:<16} {agg['wrr_mean']:>7.3f} {agg['co2_saved_kg']:>10.1f} kg")

## 8. Multi-Agent Mode: Consumer + Store Manager

In [ ]:
from freshprice_env.multi_agent_env import MultiAgentFreshPriceEnv

ma_env = MultiAgentFreshPriceEnv(
    scenario=CurriculumScenario.BUSY_WEEKEND,
    seed=42,
    consumer_price_sensitivity=1.5,
)

obs, info = ma_env.reset()
print('Multi-agent env initialized.')
print(f'Consumer signals visible in obs: {"CONSUMER SIGNALS" in obs}')

# Run 3 steps to show consumer feedback
agent = RuleBasedAgent()
for i in range(3):
    action = agent.act(obs, info)
    obs, reward, done, truncated, info = ma_env.step(action)
    consumer_obs = info.get('consumer_observation', {})
    discounts = consumer_obs.get('visible_discounts', [])
    print(f'Step {i+1}: reward={reward:.4f} | Consumer sees {len(discounts)} discounted items | '
          f'Weather={consumer_obs.get("weather","?")} | Event={consumer_obs.get("event","?")}')        

## 9. Multi-Store Cooperative Mode

In [ ]:
from freshprice_env.multi_store_env import MultiStoreFreshPriceEnv

ms_env = MultiStoreFreshPriceEnv(
    n_stores=2,
    scenario=CurriculumScenario.CRISIS_WEEK,
    seed=42,
)

obs_list, info = ms_env.reset()
print(f'Multi-store env: {info["n_stores"]} stores initialized')

# Each store acts independently
agent = RuleBasedAgent()
for step in range(3):
    actions = [agent.act(obs, {"engine_type": "PRICING"}) for obs in obs_list]
    obs_list, rewards, done, truncated, info = ms_env.step(actions)
    print(f'Step {step+1}: rewards={[f"{r:.3f}" for r in rewards]} | '
          f'collective_WRR={info["collective_wrr"]:.3f} | '
          f'transfers={info["episode_transfers_total"]}')

## 10. Impact Metrics Summary

In [ ]:
# Show the social impact story
print('=== FreshPrice AI — Social Impact Projection ===')
print()
print('Per 7-day episode (one store):')
print('  Rule-based agent:   WRR ~0.34  | CO2 saved ~18 kg | Food saved ~7 kg')
print('  Trained LLM (target): WRR ≥0.70 | CO2 saved ~38 kg | Food saved ~15 kg')
print()
print('Scale to 1,000 stores:')
import math
stores = 1000
weeks_per_year = 52
co2_per_store_week = 38.0  # kg, trained agent
food_per_store_week = 15.0
print(f'  CO2 prevented:  {stores * weeks_per_year * co2_per_store_week / 1000:.0f} tonnes/year')
print(f'  Food saved:     {stores * weeks_per_year * food_per_store_week / 1000:.0f} tonnes/year')
print(f'  WRR improvement: +{(0.70 - 0.34):.0%} vs rule-based baseline')
print()
print('This is the economic incentive: waste = lost revenue.')
print('Higher WRR = more recovered revenue = less food in landfill.')